In [0]:
%run /Workspace/Users/azuser7248_mml.local@karthikirisoutlook.onmicrosoft.com/capstone-datasets/00_config

In [0]:
import requests

gold_config_url = "https://raw.githubusercontent.com/Santhoshpec/capstone-datasets/main/configs/gold_config.json"

gold_marts_config_url = "https://raw.githubusercontent.com/Santhoshpec/capstone-datasets/main/configs/gold_marts_config.json"

gold_config = requests.get(gold_config_url).json()

gold_marts_config = requests.get(gold_marts_config_url).json()

print("Fact Config")

display(gold_config)

print("Mart Config")

display(gold_marts_config)

In [0]:
from pyspark.sql import Row
from datetime import datetime


def write_gold_audit(
        table_name,
        record_count,
        status):

    audit_path = (
        f"{base_path}/audit/gold_audit/"
    )

    audit_df = spark.createDataFrame([

        Row(

            table_name=table_name,

            run_timestamp=datetime.now(),

            record_count=record_count,

            status=status

        )

    ])

    (
        audit_df
        .write
        .format("delta")
        .mode("append")
        .save(audit_path)
    )

    print(
        f"Audit Updated : {table_name}"
    )

In [0]:
from pyspark.sql.functions import *

def create_dim_date(config):

    orders_path = (
        f"{base_path}/{config['orders_path']}/"
    )

    target_path = (
        f"{base_path}/{config['date_dimension']}/"
    )

    print("Creating Date Dimension...")

    orders_df = (
        spark.read
        .format("delta")
        .load(orders_path)
    )

    dim_date = (

        orders_df

        .select(
            col(config["date_column"]).alias("full_date")
        )

        .distinct()

        .withColumn(
            "date_key",
            date_format(
                col("full_date"),
                "yyyyMMdd"
            ).cast("int")
        )

        .withColumn(
            "year",
            year("full_date")
        )

        .withColumn(
            "quarter",
            quarter("full_date")
        )

        .withColumn(
            "month",
            month("full_date")
        )

        .withColumn(
            "month_name",
            date_format(
                "full_date",
                "MMMM"
            )
        )

        .withColumn(
            "week_of_year",
            weekofyear("full_date")
        )

        .withColumn(
            "day",
            dayofmonth("full_date")
        )

        .withColumn(
            "day_name",
            date_format(
                "full_date",
                "EEEE"
            )
        )

    )

    (
        dim_date
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
    )

    write_gold_audit(
        table_name="dim_date",
        record_count=dim_date.count(),
        status="SUCCESS"
    )

    print("Date Dimension Created Successfully")

In [0]:
from pyspark.sql.functions import *

def create_fact_sales(config):

    try:

        print(f"Creating : {config['entity']}")

        orders = (
            spark.read
            .format("delta")
            .load(
                f"{base_path}/{config['orders_path']}/"
            )
        )

        customers = (
            spark.read
            .format("delta")
            .load(
                f"{base_path}/{config['customer_dimension']}/"
            )
            .filter(
                col("is_current") == True
            )
        )

        products = (
            spark.read
            .format("delta")
            .load(
                f"{base_path}/{config['product_dimension']}/"
            )
            .filter(
                col("is_current") == True
            )
        )

        dates = (
            spark.read
            .format("delta")
            .load(
                f"{base_path}/{config['date_dimension']}/"
            )
        )

        fact_df = (

            orders.alias("o")

            .join(

                customers.alias("c"),

                col("o.CustomerID") == col("c.CustomerID"),

                "inner"

            )

            .join(

                products.alias("p"),

                col("o.ProductID") == col("p.ProductID"),

                "inner"

            )

            .join(

                dates.alias("d"),

                col("o.OrderDate") == col("d.full_date"),

                "inner"

            )

            .withColumn(

                "sales_amount",

                col("Quantity") *
                col("UnitPrice")

            )

            .select(

                monotonically_increasing_id()
                .alias("sales_key"),

                col("c.customer_sk"),

                col("p.product_sk"),

                col("d.date_key"),

                col("o.OrderID"),

                col("o.CustomerID"),

                col("o.ProductID"),

                col("o.Quantity"),

                col("o.UnitPrice"),

                col("sales_amount"),

                col("o.StoreCode")

            )

        )

        (
            fact_df
            .write
            .format("delta")
            .mode("overwrite")
            .save(
                f"{base_path}/{config['target_path']}/"
            )
        )

        write_gold_audit(

            table_name=config["entity"],

            record_count=fact_df.count(),

            status="SUCCESS"

        )

        print(
            f"{config['entity']} Created Successfully"
        )

    except Exception as e:

        write_gold_audit(

            table_name=config["entity"],

            record_count=0,

            status=f"FAILED : {str(e)}"

        )

        raise e

In [0]:
monthly_sales = (

    spark.table(
        "capstone.gold.fact_sales"
    ).alias("f")

    .join(
        spark.table(
            "capstone.gold.dim_date"
        ).alias("d"),

        "date_key"
    )

    .groupBy(
        "year",
        "month"
    )

    .agg(
        sum("sales_amount")
        .alias("total_sales")
    )

)

In [0]:
from pyspark.sql.functions import *

def create_gold_mart(config):

    try:

        print(f"Creating Mart : {config['mart_name']}")

        fact_df = (
            spark.read
            .format("delta")
            .load(
                f"{base_path}/{config['fact_path']}/"
            )
        )

        if config["dimension_path"]:

            dim_df = (
                spark.read
                .format("delta")
                .load(
                    f"{base_path}/{config['dimension_path']}/"
                )
            )

            if "is_current" in dim_df.columns:

                dim_df = (
                    dim_df
                    .filter(
                        col("is_current") == True
                    )
                )

            final_df = (

                fact_df.join(

                    dim_df,

                    config["join_key"],

                    "inner"

                )

            )

        else:

            final_df = fact_df

        mart_df = (

            final_df

            .groupBy(

                *config["group_by"]

            )

            .agg(

                sum("sales_amount")
                .alias("total_sales"),

                sum("Quantity")
                .alias("total_quantity")

            )

        )

        (

            mart_df.write

            .format("delta")

            .mode("overwrite")

            .save(

                f"{base_path}/{config['target_path']}/"

            )

        )

        write_gold_audit(

            table_name=config["mart_name"],

            record_count=mart_df.count(),

            status="SUCCESS"

        )

        print(

            f"{config['mart_name']} Created Successfully"

        )

    except Exception as e:

        write_gold_audit(

            table_name=config["mart_name"],

            record_count=0,

            status=f"FAILED : {str(e)}"

        )

        raise e

In [0]:
print("========== GOLD LAYER STARTED ==========")

# Create Date Dimension

for config in gold_config:

    create_dim_date(config)

# Create Fact Tables

for config in gold_config:

    create_fact_sales(config)

# Create Gold Marts

for mart in gold_marts_config:

    create_gold_mart(mart)

print("========== GOLD LAYER COMPLETED ==========")

In [0]:
print("========== DATE DIMENSION ==========")

display(

    spark.read

    .format("delta")

    .load(

        f"{base_path}/gold/dimensions/dim_date/"

    )

)

In [0]:
print("========== FACT SALES ==========")

display(

    spark.read

    .format("delta")

    .load(

        f"{base_path}/gold/facts/fact_sales/"

    )

)

In [0]:
print("========== SALES BY CATEGORY ==========")

display(

    spark.read

    .format("delta")

    .load(

        f"{base_path}/gold/marts/sales_by_category/"

    )

)

In [0]:
print("========== MONTHLY SALES ==========")

display(

    spark.read

    .format("delta")

    .load(

        f"{base_path}/gold/marts/monthly_sales/"

    )

)

In [0]:
print("========== SALES BY STORE ==========")

display(

    spark.read

    .format("delta")

    .load(

        f"{base_path}/gold/marts/sales_by_store/"

    )

)

In [0]:
print("========== GOLD AUDIT ==========")

display(

    spark.read

    .format("delta")

    .load(

        f"{base_path}/audit/gold_audit/"

    )

)